# Établissements Hospitaliers Franciliens - EDA & Cleaning

This notebook performs exploratory data analysis and data cleaning on the healthcare facilities dataset from Île-de-France region.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from datetime import datetime
import re

warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

# Set plot style
sns.set_style('whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

## 2. Load Data

In [ ]:
# Load the hospital data
df = pd.read_csv('data/bronze/public_service_data/les_etablissements_hospitaliers_franciliens.csv',
                 sep=';',  # French CSV often uses semicolon
                 encoding='utf-8',
                 low_memory=False)

print(f"Dataset loaded successfully!")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

## 3. Initial Data Inspection

In [ ]:
# Display first rows
print("FIRST 10 ROWS")
display(df.head(10))

In [ ]:
# Column names and types
print("COLUMN INFORMATION")
print(f"\nTotal columns: {len(df.columns)}\n")

col_info = pd.DataFrame({
    'Column': df.columns,
    'Data_Type': df.dtypes.values,
    'Non_Null': df.count().values,
    'Null_Count': df.isnull().sum().values,
    'Null_%': (df.isnull().sum().values / len(df) * 100).round(2)
})

display(col_info)

In [ ]:
# Detailed info
print("DATASET INFO")
display(df.info())

## 4. Exploratory Data Analysis (EDA)

### 4.1 Missing Values Analysis

In [ ]:
# Missing values summary
missing_summary = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': (df.isnull().sum() / len(df) * 100).round(2),
    'Data_Type': df.dtypes
}).sort_values('Missing_Count', ascending=False)

print("MISSING VALUES ANALYSIS")
print(f"\nTotal missing values: {df.isnull().sum().sum():,}\n")
display(missing_summary[missing_summary['Missing_Count'] > 0])

In [ ]:
# Visualize missing values
missing_data = missing_summary[missing_summary['Missing_Count'] > 0].sort_values('Missing_Percentage', ascending=True)

if len(missing_data) > 0:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Bar chart
    ax1.barh(missing_data['Column'], missing_data['Missing_Percentage'], color='coral')
    ax1.set_xlabel('Missing Percentage (%)', fontsize=12)
    ax1.set_title('Missing Values by Column', fontsize=14, fontweight='bold')
    ax1.grid(axis='x', alpha=0.3)
    
    # Count chart
    ax2.barh(missing_data['Column'], missing_data['Missing_Count'], color='steelblue')
    ax2.set_xlabel('Missing Count', fontsize=12)
    ax2.set_title('Missing Values Count', fontsize=14, fontweight='bold')
    ax2.grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("✅ No missing values found in the dataset!")

### 4.2 Unique Values & Cardinality

In [ ]:
# Unique values analysis
unique_summary = pd.DataFrame({
    'Column': df.columns,
    'Unique_Values': [df[col].nunique() for col in df.columns],
    'Sample_Value': [str(df[col].dropna().iloc[0]) if len(df[col].dropna()) > 0 else 'N/A' for col in df.columns]
})
unique_summary['Cardinality'] = (unique_summary['Unique_Values'] / len(df) * 100).round(2)

print("UNIQUE VALUES ANALYSIS")
display(unique_summary.sort_values('Unique_Values', ascending=False))

### 4.3 Key Categorical Variables Distribution

In [ ]:
# Department distribution
if 'dept' in df.columns:
    print("DISTRIBUTION BY DEPARTMENT (DÉPARTEMENT)")

    dept_dist = df['dept'].value_counts().sort_index()
    print(dept_dist)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Bar chart
    dept_dist.plot(kind='bar', ax=ax1, color='skyblue', edgecolor='black')
    ax1.set_title('Healthcare Facilities by Department', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Department', fontsize=12)
    ax1.set_ylabel('Number of Facilities', fontsize=12)
    ax1.tick_params(axis='x', rotation=45)
    ax1.grid(axis='y', alpha=0.3)
    
    # Pie chart
    dept_dist.plot(kind='pie', ax=ax2, autopct='%1.1f%%', startangle=90)
    ax2.set_title('Proportion by Department', fontsize=14, fontweight='bold')
    ax2.set_ylabel('')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Category of establishment
if 'categorie_de_l_etablissement' in df.columns:
    print("DISTRIBUTION BY ESTABLISHMENT CATEGORY")

    cat_dist = df['categorie_de_l_etablissement'].value_counts().head(15)
    print(cat_dist)
    
    plt.figure(figsize=(14, 8))
    cat_dist.plot(kind='barh', color='lightcoral', edgecolor='black')
    plt.title('Top 15 Establishment Categories', fontsize=14, fontweight='bold')
    plt.xlabel('Number of Facilities', fontsize=12)
    plt.ylabel('Category', fontsize=12)
    plt.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# Type of establishment
if 'type_etablissement' in df.columns:
    print("=" * 100)
    print("DISTRIBUTION BY ESTABLISHMENT TYPE")
    print("=" * 100)
    
    type_dist = df['type_etablissement'].value_counts().head(15)
    print(type_dist)
    
    plt.figure(figsize=(14, 8))
    type_dist.plot(kind='barh', color='mediumseagreen', edgecolor='black')
    plt.title('Top 15 Establishment Types', fontsize=14, fontweight='bold')
    plt.xlabel('Number of Facilities', fontsize=12)
    plt.ylabel('Type', fontsize=12)
    plt.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# Public service participation
if 'participant_service_public_hospitalier' in df.columns:
    print("=" * 100)
    print("PUBLIC HOSPITAL SERVICE PARTICIPATION")
    print("=" * 100)
    
    psph_dist = df['participant_service_public_hospitalier'].value_counts()
    print(psph_dist)
    
    plt.figure(figsize=(10, 6))
    psph_dist.plot(kind='bar', color=['#FF6B6B', '#4ECDC4', '#95E1D3'], edgecolor='black')
    plt.title('Public Hospital Service Participation', fontsize=14, fontweight='bold')
    plt.xlabel('Participation Status', fontsize=12)
    plt.ylabel('Number of Facilities', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# Tarification type
if 'lib_tarification' in df.columns:
    print("=" * 100)
    print("TARIFICATION TYPES")
    print("=" * 100)
    
    tarif_dist = df['lib_tarification'].value_counts()
    print(tarif_dist)
    
    plt.figure(figsize=(12, 6))
    tarif_dist.plot(kind='barh', color='mediumpurple', edgecolor='black')
    plt.title('Distribution by Tarification Type', fontsize=14, fontweight='bold')
    plt.xlabel('Number of Facilities', fontsize=12)
    plt.ylabel('Tarification Type', fontsize=12)
    plt.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()

### 4.4 Temporal Analysis

In [ ]:
# Opening date analysis
if 'date_ouverture' in df.columns:
    # Convert to datetime
    df['date_ouverture_dt'] = pd.to_datetime(df['date_ouverture'], errors='coerce')
    df['year_ouverture'] = df['date_ouverture_dt'].dt.year
    df['decade_ouverture'] = (df['year_ouverture'] // 10) * 10
    
    print("=" * 100)
    print("OPENING DATE STATISTICS")
    print("=" * 100)
    print(f"Earliest opening: {df['date_ouverture_dt'].min()}")
    print(f"Latest opening: {df['date_ouverture_dt'].max()}")
    print(f"Missing dates: {df['date_ouverture_dt'].isnull().sum()}")
    
    # Plot by decade
    decade_dist = df['decade_ouverture'].value_counts().sort_index()
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # By decade
    decade_dist.plot(kind='bar', ax=ax1, color='steelblue', edgecolor='black')
    ax1.set_title('Healthcare Facilities Opened by Decade', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Decade', fontsize=12)
    ax1.set_ylabel('Number of Facilities', fontsize=12)
    ax1.tick_params(axis='x', rotation=45)
    ax1.grid(axis='y', alpha=0.3)
    
    # Cumulative
    year_dist = df['year_ouverture'].value_counts().sort_index()
    cumulative = year_dist.cumsum()
    ax2.plot(cumulative.index, cumulative.values, linewidth=2, color='darkgreen')
    ax2.fill_between(cumulative.index, cumulative.values, alpha=0.3, color='lightgreen')
    ax2.set_title('Cumulative Healthcare Facilities Over Time', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Year', fontsize=12)
    ax2.set_ylabel('Cumulative Number', fontsize=12)
    ax2.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()

### 4.5 Geographic Analysis

In [ ]:
# Geographic coordinates analysis
if 'lat' in df.columns and 'lng' in df.columns:
    print("=" * 100)
    print("GEOGRAPHIC COORDINATES STATISTICS")
    print("=" * 100)
    
    # Convert to numeric
    df['lat_num'] = pd.to_numeric(df['lat'], errors='coerce')
    df['lng_num'] = pd.to_numeric(df['lng'], errors='coerce')
    
    print(f"\nLatitude:")
    print(df['lat_num'].describe())
    print(f"\nLongitude:")
    print(df['lng_num'].describe())
    print(f"\nMissing coordinates: {df['lat_num'].isnull().sum()}")
    
    # Map visualization
    fig, ax = plt.subplots(figsize=(14, 10))
    
    # Plot by department
    if 'dept' in df.columns:
        for dept in df['dept'].unique():
            dept_data = df[df['dept'] == dept]
            ax.scatter(dept_data['lng_num'], dept_data['lat_num'], 
                      alpha=0.6, s=50, label=dept, edgecolors='black', linewidth=0.5)
    else:
        ax.scatter(df['lng_num'], df['lat_num'], alpha=0.5, s=50, 
                  color='steelblue', edgecolors='black', linewidth=0.5)
    
    ax.set_xlabel('Longitude', fontsize=12)
    ax.set_ylabel('Latitude', fontsize=12)
    ax.set_title('Geographic Distribution of Healthcare Facilities in Île-de-France', 
                fontsize=14, fontweight='bold')
    ax.grid(alpha=0.3)
    if 'dept' in df.columns:
        ax.legend(title='Department', bbox_to_anchor=(1.05, 1), loc='upper left')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Heatmap by department and category
if 'dept' in df.columns and 'categorie_de_l_etablissement' in df.columns:
    # Get top categories
    top_cats = df['categorie_de_l_etablissement'].value_counts().head(10).index
    
    # Create crosstab
    heatmap_data = pd.crosstab(df['dept'], df['categorie_de_l_etablissement'])
    heatmap_data = heatmap_data[top_cats]
    
    plt.figure(figsize=(16, 8))
    sns.heatmap(heatmap_data, annot=True, fmt='d', cmap='YlOrRd', 
                cbar_kws={'label': 'Number of Facilities'})
    plt.title('Healthcare Facilities Distribution: Department × Top 10 Categories', 
             fontsize=14, fontweight='bold')
    plt.xlabel('Category', fontsize=12)
    plt.ylabel('Department', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

### 4.6 Address Analysis

In [ ]:
# Complete address availability
address_cols = ['adresse_complete', 'num_voie', 'type_voie', 'voie', 'cp_ville']
existing_addr_cols = [col for col in address_cols if col in df.columns]

if existing_addr_cols:
    print("=" * 100)
    print("ADDRESS COMPLETENESS ANALYSIS")
    print("=" * 100)
    
    for col in existing_addr_cols:
        complete = df[col].notna().sum()
        pct = (complete / len(df)) * 100
        print(f"{col:<25}: {complete:>6,} / {len(df):,} ({pct:>5.1f}%)")

In [ ]:
# Top street types
if 'type_voie' in df.columns:
    print("\n" + "=" * 100)
    print("TOP 15 STREET TYPES")
    print("=" * 100)
    
    type_voie_dist = df['type_voie'].value_counts().head(15)
    print(type_voie_dist)
    
    plt.figure(figsize=(12, 6))
    type_voie_dist.plot(kind='barh', color='teal', edgecolor='black')
    plt.title('Top 15 Street Types', fontsize=14, fontweight='bold')
    plt.xlabel('Count', fontsize=12)
    plt.ylabel('Street Type', fontsize=12)
    plt.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()

### 4.7 Statistical Summary

In [ ]:
# Numeric columns summary
numeric_cols = df.select_dtypes(include=[np.number]).columns

if len(numeric_cols) > 0:
    print("=" * 100)
    print("NUMERIC FEATURES SUMMARY")
    print("=" * 100)
    display(df[numeric_cols].describe())

### 4.8 Duplicate Analysis

In [ ]:
# Check for duplicates
print("=" * 100)
print("DUPLICATE ANALYSIS")
print("=" * 100)

# Full row duplicates
full_dupes = df.duplicated().sum()
print(f"\nFull row duplicates: {full_dupes:,}")

# FINESS duplicates (should be unique)
if 'finess_et' in df.columns:
    finess_dupes = df['finess_et'].duplicated().sum()
    print(f"Duplicate FINESS ET codes: {finess_dupes:,}")
    
    if finess_dupes > 0:
        print("\nDuplicate FINESS ET examples:")
        dupe_finess = df[df['finess_et'].duplicated(keep=False)].sort_values('finess_et')
        display(dupe_finess[['finess_et', 'raison_sociale', 'adresse_complete', 'dept']].head(10))

# Address duplicates
if 'adresse_complete' in df.columns:
    addr_dupes = df['adresse_complete'].duplicated().sum()
    print(f"\nDuplicate addresses: {addr_dupes:,}")

## 5. Data Cleaning

In [ ]:
# Create a copy for cleaning
df_clean = df.copy()

print("=" * 100)
print("STARTING DATA CLEANING PROCESS")
print("=" * 100)
print(f"\nInitial shape: {df_clean.shape}")
print(f"Initial memory: {df_clean.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

### 5.1 Remove Full Duplicates

In [ ]:
# Remove complete duplicate rows
initial_rows = len(df_clean)
df_clean = df_clean.drop_duplicates()
removed_dupes = initial_rows - len(df_clean)

print(f"Removed {removed_dupes:,} duplicate rows")
print(f"Remaining rows: {len(df_clean):,}")

### 5.2 Handle FINESS Duplicates

In [ ]:
# Keep only the first occurrence of duplicate FINESS codes
if 'finess_et' in df_clean.columns:
    initial_rows = len(df_clean)
    df_clean = df_clean.drop_duplicates(subset=['finess_et'], keep='first')
    removed_finess_dupes = initial_rows - len(df_clean)
    
    print(f"\nRemoved {removed_finess_dupes:,} duplicate FINESS ET records (kept first occurrence)")
    print(f"Remaining rows: {len(df_clean):,}")

### 5.3 Data Type Corrections

In [ ]:
# Fix data types
print("\n" + "="*100)
print("CORRECTING DATA TYPES")
print("="*100)

# FINESS codes (keep as string with leading zeros)
for col in ['finess_et', 'finess_ej']:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype(str).str.strip()
        print(f"✓ {col}: converted to string")

# Numeric codes
numeric_code_cols = ['num_dept', 'num_cat', 'num_type', 'code_psph', 'code_tarif']
for col in numeric_code_cols:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
        print(f"✓ {col}: converted to numeric")

# Coordinates
for col in ['lat', 'lng']:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
        print(f"✓ {col}: converted to float")

# Date
if 'date_ouverture' in df_clean.columns:
    df_clean['date_ouverture'] = pd.to_datetime(df_clean['date_ouverture'], errors='coerce')
    print(f"✓ date_ouverture: converted to datetime")

# Postal code
if 'cp_ville' in df_clean.columns:
    # Extract postal code (first 5 digits)
    df_clean['code_postal'] = df_clean['cp_ville'].str.extract(r'(\d{5})')[0]
    print(f"✓ code_postal: extracted from cp_ville")

### 5.4 Clean String Data

In [ ]:
# Clean all string columns
print("\n" + "="*100)
print("CLEANING STRING DATA")
print("="*100)

string_cols = df_clean.select_dtypes(include=['object']).columns

for col in string_cols:
    if col not in ['date_ouverture']:  # Skip already converted columns
        # Strip whitespace
        df_clean[col] = df_clean[col].astype(str).str.strip()
        
        # Remove multiple spaces
        df_clean[col] = df_clean[col].str.replace(r'\s+', ' ', regex=True)
        
        # Replace 'nan' string with actual NaN
        df_clean.loc[df_clean[col] == 'nan', col] = np.nan

print(f"✓ Cleaned {len(string_cols)} string columns")

### 5.5 Handle Missing Values

In [ ]:
print("\n" + "="*100)
print("HANDLING MISSING VALUES")
print("="*100)

# Strategy:
# 1. Drop columns with >70% missing (if not essential)
# 2. Fill categorical with 'Non renseigné'
# 3. Keep numeric NaN as is (or fill with appropriate value)

missing_threshold = 70

# Identify columns to potentially drop
missing_pct = (df_clean.isnull().sum() / len(df_clean)) * 100
cols_high_missing = missing_pct[missing_pct > missing_threshold].index.tolist()

# Essential columns to keep regardless of missing values
essential_cols = ['finess_et', 'finess_ej', 'raison_sociale', 'lat', 'lng', 
                 'dept', 'categorie_de_l_etablissement', 'type_etablissement']

cols_to_drop = [col for col in cols_high_missing if col not in essential_cols]

if cols_to_drop:
    print(f"\nDropping {len(cols_to_drop)} columns with >{missing_threshold}% missing:")
    for col in cols_to_drop:
        pct = missing_pct[col]
        print(f"  - {col}: {pct:.1f}% missing")
    df_clean = df_clean.drop(cols_to_drop, axis=1)
else:
    print(f"\nNo columns exceed {missing_threshold}% missing threshold")

# Fill categorical missing values
categorical_fill_cols = ['adresse_administrative_2', 'complement_adresse', 'num_fax']
for col in categorical_fill_cols:
    if col in df_clean.columns:
        df_clean[col].fillna('Non renseigné', inplace=True)
        print(f"✓ Filled '{col}' with 'Non renseigné'")

# Fill numeric street number
if 'num_voie' in df_clean.columns:
    df_clean['num_voie'].fillna(None, inplace=True)
    print(f"✓ Filled 'num_voie' with empty string")

### 5.6 Create Derived Features

In [ ]:
print("\n" + "="*100)
print("CREATING DERIVED FEATURES")
print("="*100)

# Extract year, decade from opening date
if 'date_ouverture' in df_clean.columns:
    df_clean['annee_ouverture'] = df_clean['date_ouverture'].dt.year
    df_clean['decennie_ouverture'] = (df_clean['annee_ouverture'] // 10) * 10
    df_clean['age_etablissement'] = 2024 - df_clean['annee_ouverture']
    print("✓ Created: annee_ouverture, decennie_ouverture, age_etablissement")

# Create full address if not exists
if 'adresse_complete' not in df_clean.columns or df_clean['adresse_complete'].isnull().sum() > 0:
    if all(col in df_clean.columns for col in ['num_voie', 'type_voie', 'voie']):
        df_clean['adresse_reconstituee'] = (
            df_clean['num_voie'].astype(str) + ' ' + 
            df_clean['type_voie'].astype(str) + ' ' + 
            df_clean['voie'].astype(str)
        ).str.strip()
        print("✓ Created: adresse_reconstituee")

# Department number
if 'code_postal' in df_clean.columns:
    df_clean['num_dept_from_cp'] = df_clean['code_postal'].str[:2]
    print("✓ Created: num_dept_from_cp")

# Binary flags
if 'participant_service_public_hospitalier' in df_clean.columns:
    df_clean['is_public_service'] = df_clean['participant_service_public_hospitalier'].apply(
        lambda x: 1 if x == 'Etablissement public de santé' else 0
    )
    print("✓ Created: is_public_service (binary flag)")

# Has coordinates flag
if 'lat' in df_clean.columns and 'lng' in df_clean.columns:
    df_clean['has_coordinates'] = df_clean['lat'].notna() & df_clean['lng'].notna()
    print("✓ Created: has_coordinates (binary flag)")

### 5.7 Standardize Categorical Values

In [ ]:
print("\n" + "="*100)
print("STANDARDIZING CATEGORICAL VALUES")
print("="*100)

# Standardize department names (capitalize)
if 'dept' in df_clean.columns:
    df_clean['dept'] = df_clean['dept'].str.upper()
    print("✓ Standardized department names to uppercase")

# Standardize establishment names (title case)
if 'raison_sociale' in df_clean.columns:
    # Keep original but clean
    df_clean['raison_sociale'] = df_clean['raison_sociale'].str.strip()
    print("✓ Cleaned establishment names")

### 5.8 Validate Coordinates

In [ ]:
# Validate geographic coordinates for Île-de-France region
print("\n" + "="*100)
print("VALIDATING GEOGRAPHIC COORDINATES")
print("="*100)

if 'lat' in df_clean.columns and 'lng' in df_clean.columns:
    # Île-de-France approximate bounds
    lat_min, lat_max = 48.1, 49.3
    lng_min, lng_max = 1.4, 3.6
    
    # Find invalid coordinates
    invalid_coords = df_clean[
        (df_clean['lat'].notna()) & (df_clean['lng'].notna()) &
        ((df_clean['lat'] < lat_min) | (df_clean['lat'] > lat_max) |
         (df_clean['lng'] < lng_min) | (df_clean['lng'] > lng_max))
    ]
    
    print(f"Found {len(invalid_coords)} facilities with coordinates outside Île-de-France bounds")
    
    if len(invalid_coords) > 0:
        print("\nInvalid coordinate examples:")
        display(invalid_coords[['finess_et', 'raison_sociale', 'lat', 'lng', 'dept']].head())
        
        # Optionally set invalid coordinates to NaN
        # df_clean.loc[invalid_coords.index, ['lat', 'lng']] = np.nan
    else:
        print("✓ All coordinates are within valid Île-de-France bounds")

### 5.9 Final Quality Check

In [ ]:
print("\n" + "="*100)
print("FINAL DATA QUALITY REPORT")
print("="*100)

print(f"\n📊 Dataset Dimensions:")
print(f"  Rows: {len(df_clean):,}")
print(f"  Columns: {df_clean.shape[1]}")
print(f"  Memory: {df_clean.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print(f"\n🔍 Missing Values:")
total_missing = df_clean.isnull().sum().sum()
print(f"  Total: {total_missing:,}")

if total_missing > 0:
    print(f"\n  By Column:")
    missing_by_col = df_clean.isnull().sum()
    missing_by_col = missing_by_col[missing_by_col > 0].sort_values(ascending=False)
    for col, count in missing_by_col.items():
        pct = (count / len(df_clean)) * 100
        print(f"    {col:<40}: {count:>6,} ({pct:>5.1f}%)")

print(f"\n🔄 Duplicates:")
print(f"  Full row duplicates: {df_clean.duplicated().sum()}")
if 'finess_et' in df_clean.columns:
    print(f"  Duplicate FINESS codes: {df_clean['finess_et'].duplicated().sum()}")

print(f"\n📍 Geographic Coverage:")
if 'lat' in df_clean.columns and 'lng' in df_clean.columns:
    coords_available = (df_clean['lat'].notna() & df_clean['lng'].notna()).sum()
    coords_pct = (coords_available / len(df_clean)) * 100
    print(f"  Facilities with coordinates: {coords_available:,} ({coords_pct:.1f}%)")

if 'dept' in df_clean.columns:
    print(f"  Departments covered: {df_clean['dept'].nunique()}")
    print(f"  Distribution: {df_clean['dept'].value_counts().to_dict()}")

In [ ]:
# Display cleaned data sample
print("\n" + "="*100)
print("CLEANED DATA SAMPLE")
print("="*100)
display(df_clean.head(10))

## 6. Save Cleaned Data

In [ ]:
# Save cleaned data
print("="*100)
print("SAVING CLEANED DATA")
print("="*100)

# CSV
output_csv = '/data/les_etablissements_hospitaliers_franciliens_cleaned.csv'
df_clean.to_csv(output_csv, index=False, sep=';', encoding='utf-8-sig')
print(f"✓ Saved to CSV: {output_csv}")

# Excel
output_excel = '/data/les_etablissements_hospitaliers_franciliens_cleaned.xlsx'
df_clean.to_excel(output_excel, index=False, engine='openpyxl')
print(f"✓ Saved to Excel: {output_excel}")

# GeoJSON (if coordinates available)
if 'lat' in df_clean.columns and 'lng' in df_clean.columns:
    try:
        import json
        
        # Filter only records with valid coordinates
        df_geo = df_clean[df_clean['lat'].notna() & df_clean['lng'].notna()].copy()
        
        # Create GeoJSON structure
        features = []
        for idx, row in df_geo.iterrows():
            feature = {
                "type": "Feature",
                "geometry": {
                    "type": "Point",
                    "coordinates": [float(row['lng']), float(row['lat'])]
                },
                "properties": {
                    "finess_et": str(row['finess_et']),
                    "raison_sociale": str(row['raison_sociale']),
                    "categorie": str(row.get('categorie_de_l_etablissement', '')),
                    "type": str(row.get('type_etablissement', '')),
                    "dept": str(row.get('dept', '')),
                    "adresse": str(row.get('adresse_complete', ''))
                }
            }
            features.append(feature)
        
        geojson = {
            "type": "FeatureCollection",
            "features": features
        }
        
        output_geojson = '/data/les_etablissements_hospitaliers_franciliens.geojson'
        with open(output_geojson, 'w', encoding='utf-8') as f:
            json.dump(geojson, f, ensure_ascii=False, indent=2)
        
        print(f"✓ Saved to GeoJSON: {output_geojson}")
        print(f"  ({len(features):,} facilities with coordinates)")
    except Exception as e:
        print(f"⚠ Could not create GeoJSON: {e}")

## 7. Generate Data Dictionary

In [ ]:
# Create comprehensive data dictionary
data_dict = pd.DataFrame({
    'Column_Name': df_clean.columns,
    'Data_Type': df_clean.dtypes.astype(str).values,
    'Non_Null_Count': df_clean.count().values,
    'Null_Count': df_clean.isnull().sum().values,
    'Null_Percentage': (df_clean.isnull().sum() / len(df_clean) * 100).round(2).values,
    'Unique_Values': [df_clean[col].nunique() for col in df_clean.columns],
    'Sample_Value': [str(df_clean[col].dropna().iloc[0]) if len(df_clean[col].dropna()) > 0 else 'N/A' 
                     for col in df_clean.columns]
})

# Add descriptions (French)
descriptions = {
    'finess_et': 'Numéro FINESS de l\'établissement',
    'finess_ej': 'Numéro FINESS de l\'entité juridique',
    'raison_sociale': 'Nom de l\'établissement',
    'dept': 'Département',
    'categorie_de_l_etablissement': 'Catégorie de l\'établissement',
    'type_etablissement': 'Type d\'établissement',
    'participant_service_public_hospitalier': 'Participation au service public hospitalier',
    'date_ouverture': 'Date d\'ouverture',
    'lat': 'Latitude (WGS84)',
    'lng': 'Longitude (WGS84)',
    'adresse_complete': 'Adresse complète',
    'cp_ville': 'Code postal et ville'
}

data_dict['Description'] = data_dict['Column_Name'].map(descriptions).fillna('')

# Save data dictionary
dict_path = '/data/etablissements_hospitaliers_data_dictionary.csv'
data_dict.to_csv(dict_path, index=False, encoding='utf-8-sig')
print(f"\n✓ Data dictionary saved: {dict_path}")

display(data_dict)

## 8. Summary Statistics

In [ ]:
# Generate summary report
print("="*100)
print("FINAL SUMMARY STATISTICS")
print("="*100)

summary = {
    'Metric': [
        'Total Healthcare Facilities',
        'Total Features/Columns',
        'Numeric Features',
        'Categorical Features',
        'Departments Covered',
        'Establishment Categories',
        'Establishment Types',
        'Facilities with Coordinates',
        'Public Service Participants',
        'Oldest Facility Opening',
        'Newest Facility Opening',
        'Total Missing Values',
        'Data Completeness',
        'Memory Usage (MB)'
    ],
    'Value': [
        f"{len(df_clean):,}",
        df_clean.shape[1],
        len(df_clean.select_dtypes(include=[np.number]).columns),
        len(df_clean.select_dtypes(include=['object']).columns),
        df_clean['dept'].nunique() if 'dept' in df_clean.columns else 'N/A',
        df_clean['categorie_de_l_etablissement'].nunique() if 'categorie_de_l_etablissement' in df_clean.columns else 'N/A',
        df_clean['type_etablissement'].nunique() if 'type_etablissement' in df_clean.columns else 'N/A',
        f"{(df_clean['lat'].notna() & df_clean['lng'].notna()).sum():,}" if 'lat' in df_clean.columns else 'N/A',
        f"{(df_clean['is_public_service']==1).sum():,}" if 'is_public_service' in df_clean.columns else 'N/A',
        str(df_clean['date_ouverture'].min()) if 'date_ouverture' in df_clean.columns else 'N/A',
        str(df_clean['date_ouverture'].max()) if 'date_ouverture' in df_clean.columns else 'N/A',
        f"{df_clean.isnull().sum().sum():,}",
        f"{((1 - df_clean.isnull().sum().sum() / (len(df_clean) * df_clean.shape[1])) * 100):.2f}%",
        f"{df_clean.memory_usage(deep=True).sum() / 1024**2:.2f}"
    ]
}

summary_df = pd.DataFrame(summary)
display(summary_df)

# Save summary
summary_path = 'etablissements_hospitaliers_summary.csv'
summary_df.to_csv(summary_path, index=False, encoding='utf-8-sig')
print(f"\n✓ Summary saved: {summary_path}")

In [ ]:
# Save to csv
summary_path = 'etablissements_hospitaliers_summary.csv'
summary_df.to_csv(summary_path, index=False, encoding='utf-8-sig')